In [42]:
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:90% !important; }</style>"))
import numpy as np
import pandas as pd
import h5py
import os
from tqdm import tqdm 
from scipy import stats

/var/folders/fc/24x3k2m92bvbv7ck1v5mt3n40000gn/T/ipykernel_34111/4217363352.py:1: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML


In [2]:
## helper functions for analysis 
#alignment with ezTrack location tracking data
def alignMiniscopeBehavCamTimestamps(savePath, sessionPath, behavCamFrameRate, miniscopeCamFrameRate):
    #'\\'.join(sessionPath.split(os.sep)[:-1])+'\\timestamp.dat'
    print(sessionPath.split(os.sep)[:-1])
    # load eZ track output and behavior camera timestamps from miniscope software 
    ezTrackOutput = pd.read_csv(sessionPath)
    timestampfile = pd.read_table('/'.join(sessionPath.split(os.sep)[:-1])+'/timeStamps.csv', delimiter=',')
    miniscope_timestampfile = pd.read_table('/'.join(sessionPath.split(os.sep)[:-1])+'/timeStampsMiniscope.csv', delimiter=',')
    
    timestampfile_td = timestampfile.set_index(pd.to_timedelta(np.linspace(0, (len(timestampfile)-1)*(1/behavCamFrameRate), len(timestampfile)), unit='s'), drop=False)
  
    miniscopetimestamp_td = miniscope_timestampfile.set_index(pd.to_timedelta(np.linspace(0, (len(miniscope_timestampfile)-1)*(1/miniscopeCamFrameRate), len(miniscope_timestampfile)), unit='s'), drop=False)
    
    behavCam_frames = []
    sys_clock_behavCam = []
    #create "key" for aligning miniscope frames to timestamp file
    #then create behavior TD and align
    for msCam_frame in tqdm(range(0, len(miniscopetimestamp_td['Frame Number']))):
        #get sys clock time of each miniscope recorded frame
        #sys_clock_msCam = time_stamps['sysClock'].loc[msCam_frame]
        #find behav cam frame closest to sys clock time of ms frame
        behavCam_frame = list(timestampfile_td.iloc[(timestampfile_td['Time Stamp (ms)']-miniscopetimestamp_td['Time Stamp (ms)'].iloc[msCam_frame]).abs().argsort()[:1]].index)[0]
        #this is the behavCamIndex that is closest to the corresponding miniscope frame 
        behavCam_frames.append(behavCam_frame)
        sys_clock_behavCam.append(timestampfile_td.loc[behavCam_frame]['Time Stamp (ms)'])

    behavCamIdxToAlign = [timestampfile_td.index.get_loc(idx) for idx in behavCam_frames]
    #ezTrackOutput

    miniscopetimestamp_td['closestBehavCamFrameIdx'] = behavCamIdxToAlign

    X_coor=[]
    Y_coor=[]
    Distance_px=[] 

    for i in miniscopetimestamp_td['closestBehavCamFrameIdx'].values:
        X_coor.append(ezTrackOutput.loc[miniscopetimestamp_td['closestBehavCamFrameIdx'].values[i]]['X'])
        Y_coor.append(ezTrackOutput.loc[miniscopetimestamp_td['closestBehavCamFrameIdx'].values[i]]['Y'])
        Distance_px.append(ezTrackOutput.loc[miniscopetimestamp_td['closestBehavCamFrameIdx'].values[i]]['Distance_px'])
    
    miniscopetimestamp_td['X_coor'] = X_coor
    miniscopetimestamp_td['Y_coor'] = Y_coor
    miniscopetimestamp_td['Distance_px'] = Distance_px
    
    return(miniscopetimestamp_td)

In [43]:
## load and do some preprocessing on the CNMFE traces 
def normalize(trace, percentile=True):
    """ Normalize a fluorescence trace by its max or its 99th percentile. """
    trace = trace - np.min(trace)
    if np.percentile(trace, 99) > 0:
        if percentile:
            trace = trace / np.percentile(trace, 99)
        else:
            trace = trace / np.max(trace)
    return trace
    
def getCellTracesInscopix(dir_path, file_name_calcium_traces, file_name_properties):

    CNMFE_raw = pd.read_csv(dir_path+file_name_calcium_traces)
    
    
    time_column = pd.to_numeric(CNMFE_raw.iloc[1:,0], errors='coerce')  # Convert to numeric, ignoring potential errors
    time_deltas = pd.to_timedelta(time_column, unit='s')
    time_deltas
    CNMFE_raw.columns = range(CNMFE_raw.shape[1])
    CNMFE_real_cells = CNMFE_raw.loc[:, CNMFE_raw.iloc[0].str.strip() == 'accepted'].copy()
    CNMFE_real_cells = CNMFE_real_cells.drop(index=0).reset_index(drop=True)
    CNMFE_real_cells.index = time_deltas
    # Convert all columns in CNMFE_real_cells to numeric, coercing errors to NaN
    CNMFE_real_cells = CNMFE_real_cells.apply(pd.to_numeric, errors='coerce')
    C_normalized = CNMFE_real_cells.apply(lambda col: normalize(col), axis=0)
    C_z_scored = CNMFE_real_cells.apply(stats.zscore).set_index(pd.to_timedelta(np.linspace(0, (len(CNMFE_real_cells)-1)*(1/20), len(CNMFE_real_cells)), unit='s'), drop=True)
    C_normalized_z_scored = C_normalized.apply(stats.zscore).set_index(pd.to_timedelta(np.linspace(0, (len(C_normalized)-1)*(1/20), len(C_normalized)), unit='s'), drop=True)

    ##load spatial components by session
    # for v4 dimensions are 600x600 pixels
    cellProps = pd.read_csv(dir_path+file_name_properties)
    cellProps = pd.read_csv(dir_path+'cell_traces_mouse1day1-props copy.csv')
    cellProps = cellProps.rename(columns={'CentroidX': 'x', 'CentroidY': 'y'})
    cellProps
    com_df = cellProps[['x', 'y']]
    com_df
 
    C_normalized_z_scored.to_csv(dir_path+file_name.strip(".csv")+'_C_traces_filtered_origHz.csv')
    com_df.to_csv(dir_path+file_name.strip(".csv")+'_com_filtered.csv')
    
    print('finished, saved:')
    print(file_name)
    
    return(C_normalized_z_scored, com_df)

In [45]:
#behavior analysis info
savePath = r'/Users/johnmarshall/Documents/Analysis/miniscope_lineartrack/m1d1_09_10_35'
sessionPath = r'/Users/johnmarshall/Documents/Analysis/miniscope_lineartrack/m1d1_09_10_35/Mouse1Day1_custom_cropped_output_LocationOutput.csv'
behavCamFrameRate = 15
miniscopeCamFrameRate = 20 

behavCamDataAligned = alignMiniscopeBehavCamTimestamps(savePath, sessionPath, behavCamFrameRate, miniscopeCamFrameRate)

#calcium analysis data
dir_path = r'/Users/johnmarshall/Documents/Analysis/miniscope_lineartrack/m1d1_09_10_35/'
file_name_ca_traces = 'cell_traces_mouse1day1 copy.csv'
file_name_props = 'cell_traces_mouse1day1-props copy.csv'
C_normalized_z_scored, com_df = getCellTracesInscopix(dir_path, file_name_ca_traces, file_name_props)

# align tracking data to CNMFE for the movies we've analyzed 
CNMFE_aligned = pd.concat([C_normalized_z_scored, behavCamDataAligned.iloc[0:len(C_normalized_z_scored)]], axis=1)
CNMFE_aligned.to_csv(dir_path+file_name_ca_traces.strip(".csv")+'cellTracesAlignedToTracking.csv')

['', 'Users', 'johnmarshall', 'Documents', 'Analysis', 'miniscope_lineartrack', 'm1d1_09_10_35']


100%|█████████████████████████████████████████████████████████████████████| 11844/11844 [00:03<00:00, 3947.71it/s]
/var/folders/fc/24x3k2m92bvbv7ck1v5mt3n40000gn/T/ipykernel_34111/482345979.py:14: DtypeWarning: Columns (0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174,175,176,177,178,179,180,181,182,183,184,185,186,187,188,189,190,191,192,193,194,195,196,197,198,199,200,201,202,203,204,205,206,207,208,209,210,211,212,213,214,215,216,217,218,219,220,221,22

finished, saved:
cell_traces_mouse1day1 copy.csv


In [46]:
CNMFE_aligned

,25,37,50,51,53,55,63,68,72,73,...,1279,1280,1288,Frame Number,Time Stamp (ms),Buffer Index,closestBehavCamFrameIdx,X_coor,Y_coor,Distance_px
0 days 00:00:00,2.644418,-2.558454,0.857165,-0.358184,-1.028813,-1.744231,-0.840018,-0.708398,-0.959497,-0.310010,...,-0.812794,-0.689902,-0.656659,0,-2,0,1,16.759387,29.402039,1.909693
0 days 00:00:00.050000,2.712025,-2.754259,0.918060,-0.210707,-0.810009,-1.523159,-0.926973,0.476910,-0.822385,-0.123732,...,-0.812794,-0.689902,-0.623503,1,53,0,1,16.759387,29.402039,1.909693
0 days 00:00:00.100000,2.571179,-2.750833,1.216193,-0.317268,-0.901976,-1.441307,-0.656137,0.190377,-0.791954,-0.163323,...,-0.812794,-0.689902,-0.624841,2,101,0,2,17.171553,30.147997,0.852253
0 days 00:00:00.150000,2.647205,-2.398786,0.869532,-0.333523,-1.020620,-2.023957,-1.224260,-0.052093,-1.027420,-0.153036,...,-0.812794,-0.689902,-0.602977,3,151,0,3,16.535266,29.625703,0.823196
0 days 00:00:00.200000,2.732407,-2.418443,1.217441,-0.364400,-1.122714,-1.772474,-0.943409,-0.483600,-0.967844,-0.300710,...,-0.812794,-0.689902,-0.501112,4,202,0,4,16.533927,29.710669,0.084976
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
0 days 00:09:09.750000,-0.362983,0.688576,0.067871,0.090784,-1.104562,-0.082399,-0.931345,0.405477,0.576371,-0.394078,...,0.847672,-0.232573,-0.581064,10995,556531,0,7979,637.474418,11.869364,0.226191
0 days 00:09:09.800000,-0.377010,0.551287,-0.043058,-0.300367,-1.147201,-0.302582,-0.656502,-0.630553,0.615889,-0.639870,...,0.779022,-0.253580,-0.584134,10996,556581,0,7980,636.943301,12.005045,0.548174
0 days 00:09:09.850000,-0.625979,0.148120,-0.013416,0.097377,-0.899631,-0.106326,-0.309650,-0.004028,0.669098,-0.760710,...,0.713211,-0.273621,-0.587078,10997,556631,0,7980,636.943301,12.005045,0.548174
0 days 00:09:09.900000,-0.358999,0.519628,0.059525,-0.241471,-1.003899,-0.395431,-0.089371,0.130469,0.664829,-0.473592,...,0.650121,-0.292741,-0.589904,10998,556681,0,7981,636.943301,12.005045,0.548174
